# Notebook 03 — The Spatial Join: Linking Births to Environmental Data

**GeoAI for Climate, Environment and Health — Zimbabwe DHS 2015 Workshop**  
25 June 2026 · YSN Bulawayo

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Implement a spatial join linking DHS clusters to a monthly climate grid using `scipy.spatial.cKDTree`.
2. Explain the DHS cluster jitter protocol and quantify its implications for spatial uncertainty in exposure assignment.
3. Perform nearest-neighbour matching and compute approximate inter-point distances in kilometres.
4. Assess cluster-level spatial associations between heat exposure and low birth weight.

---

**Context.** In environmental epidemiology, the spatial join is the critical step where individual health records acquire environmental exposure values. DHS clusters have deliberately displaced coordinates — 0–2 km for urban clusters, 0–5 km for rural clusters, and up to 10 km for approximately 1% of rural clusters — to protect respondent anonymity. This means exposures can only be assigned to the cluster centroid, not to individual women, and all downstream estimates carry inherent positional uncertainty that must be acknowledged.

---
## Section 1 — Load Data

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.spatial import cKDTree
from scipy import stats

DATA_DIR = pathlib.Path().resolve().parent / 'data'

births     = pd.read_csv(DATA_DIR / 'zw_births.csv')
climate    = pd.read_csv(DATA_DIR / 'zw_climate_monthly.csv')
clusters   = pd.read_csv(DATA_DIR / 'zw_clusters.csv')
facilities = pd.read_csv(DATA_DIR / 'health_facilities.csv')

print(f"Births:      {births.shape[0]:,} rows, {births['cluster_id'].nunique()} unique clusters")
print(f"Climate:     {climate.shape[0]:,} rows  "
      f"({climate['cluster_id'].nunique()} clusters × "
      f"{climate.groupby('cluster_id').size().iloc[0]} months)")
print(f"Clusters:    {clusters.shape[0]} rows")
print(f"Facilities:  {facilities.shape[0]} rows")

In [ ]:
# Annual mean climate per cluster — average all months, 2011–2015
# This is the "exposure grid" we will join to point locations.
clim_annual = (
    climate
    .groupby('cluster_id')[['lst_day_c', 'chirps_mm', 'ndvi', 'heat_index']]
    .mean()
    .reset_index()
    .rename(columns={
        'lst_day_c':   'mean_lst',
        'chirps_mm':   'mean_rain',
        'ndvi':        'mean_ndvi',
        'heat_index':  'mean_heat_index'
    })
)

# Attach cluster coordinates for spatial operations
clim_annual = clim_annual.merge(
    clusters[['cluster_id', 'cluster_lat', 'cluster_lon', 'province', 'urban']],
    on='cluster_id'
)

print(f"Exposure grid: {len(clim_annual)} clusters")
print(clim_annual[['mean_lst', 'mean_rain', 'mean_ndvi']].describe().round(2))

---
## Section 2 — Worked Example: Spatial Join with cKDTree

We demonstrate `cKDTree` by finding the nearest DHS cluster for each health facility. This mirrors the general use-case: assigning environmental data extracted at cluster centroids to any arbitrary point of interest.

In [ ]:
def haversine_approx_km(lat1, lon1, lat2, lon2):
    """Flat-earth approximation. Accurate to <0.5% for distances <200 km."""
    mid_lat  = np.radians((np.asarray(lat1) + np.asarray(lat2)) / 2.0)
    dlat_km  = (np.asarray(lat2) - np.asarray(lat1)) * 111.0
    dlon_km  = (np.asarray(lon2) - np.asarray(lon1)) * 111.0 * np.cos(mid_lat)
    return np.sqrt(dlat_km**2 + dlon_km**2)

In [ ]:
# Build k-d tree on cluster coordinates (lat, lon)
cluster_coords = clim_annual[['cluster_lat', 'cluster_lon']].values
tree = cKDTree(cluster_coords)

# Query: for each facility, find the nearest DHS cluster
facility_coords = facilities[['latitude', 'longitude']].values
_, nn_idx = tree.query(facility_coords, k=1)

nn_rows = clim_annual.iloc[nn_idx].reset_index(drop=True)

result = pd.DataFrame({
    'facility_id'     : facilities['facility_id'].values,
    'facility_type'   : facilities['facility_type'].values,
    'fac_province'    : facilities['province'].values,
    'nearest_cluster' : nn_rows['cluster_id'].values,
    'cluster_province': nn_rows['province'].values,
    'distance_km'     : haversine_approx_km(
        facilities['latitude'].values,  facilities['longitude'].values,
        nn_rows['cluster_lat'].values,  nn_rows['cluster_lon'].values
    ).round(2)
})

print("Facility → nearest DHS cluster (first 10 rows):")
print(result.head(10).to_string(index=False))
print()
print(f"Median facility-to-cluster distance: {result['distance_km'].median():.1f} km")
print(f"Max    facility-to-cluster distance: {result['distance_km'].max():.1f} km")

---
## Section 3 — Exercise 1 (3 pts): Leave-One-Out Nearest-Neighbour Distances

For each DHS cluster, find the nearest *other* cluster (excluding itself). Compute the distribution of inter-cluster distances.

**Store:**
- `median_nn_dist_km` — float, 1 decimal place
- `max_nn_dist_km`    — float, 1 decimal place

Plot a histogram of inter-cluster distances. This analysis reveals the effective spatial resolution of the DHS sampling frame and informs how much the 10 km jitter can "bleed" an observation across cluster boundaries.

In [ ]:
### BEGIN SOLUTION

# k=2: first neighbour is the point itself (distance = 0); second is the nearest *other* cluster
_, nn_idxs = tree.query(cluster_coords, k=2)

lat_arr = clim_annual['cluster_lat'].values
lon_arr = clim_annual['cluster_lon'].values

nn_km = haversine_approx_km(
    lat_arr,               lon_arr,
    lat_arr[nn_idxs[:, 1]], lon_arr[nn_idxs[:, 1]]
)

median_nn_dist_km = round(float(np.median(nn_km)), 1)
max_nn_dist_km    = round(float(np.max(nn_km)), 1)

print(f"Median nearest-neighbour distance: {median_nn_dist_km} km")
print(f"Max    nearest-neighbour distance: {max_nn_dist_km} km")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(nn_km, bins=30, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(median_nn_dist_km, color='firebrick', lw=1.8, linestyle='--',
           label=f'Median = {median_nn_dist_km} km')
ax.axvline(10, color='darkorange', lw=1.4, linestyle=':',
           label='DHS max jitter = 10 km')
ax.set_xlabel('Distance to nearest other cluster (km)', fontsize=11)
ax.set_ylabel('Number of clusters', fontsize=11)
ax.set_title('Inter-cluster nearest-neighbour distances — Zimbabwe DHS 2015', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

### END SOLUTION

**Interpretation (discuss with your group).** If the median nearest-neighbour distance substantially exceeds the DHS jitter radius (up to 10 km), then displaced centroids are unlikely to fall inside a neighbouring cluster's territory. If clusters are densely packed, jitter causes spatial misclassification: the displaced centroid absorbs the environmental signal of a different cluster, adding non-differential measurement error to any exposure–outcome association.

In [ ]:
### BEGIN HIDDEN TESTS
assert isinstance(median_nn_dist_km, float), "median_nn_dist_km must be a float"
assert isinstance(max_nn_dist_km, float),    "max_nn_dist_km must be a float"
assert 3.0 <= median_nn_dist_km <= 80.0,     "median inter-cluster distance out of plausible range"
assert max_nn_dist_km > median_nn_dist_km,   "max must exceed median"
assert len(nn_km) == len(clusters),          "should compute one distance per cluster"
### END HIDDEN TESTS

---
## Section 4 — Worked Example: The Jitter Uncertainty Envelope

DHS coordinates are displaced to protect respondent privacy. We simulate the displacement for rural clusters and measure how much the assigned exposure can vary purely as a function of positional uncertainty.

In [ ]:
np.random.seed(42)

N_SIM        = 100    # displaced positions simulated per cluster
RURAL_JITTER = 5.0   # km — DHS maximum displacement for rural clusters
DEG_PER_KM   = 1.0 / 111.0

rural_mask     = clim_annual['urban'] == 0
rural_clusters = clim_annual[rural_mask].reset_index(drop=True)
chosen         = rural_clusters.sample(10, random_state=42).reset_index(drop=True)

jitter_results = []
for _, row in chosen.iterrows():
    angles   = np.random.uniform(0, 2 * np.pi, N_SIM)
    radii_km = np.random.uniform(0, RURAL_JITTER, N_SIM)
    jit_lat  = row['cluster_lat'] + radii_km * np.cos(angles) * DEG_PER_KM
    jit_lon  = row['cluster_lon'] + radii_km * np.sin(angles) * DEG_PER_KM

    _, idx = tree.query(np.column_stack([jit_lat, jit_lon]), k=1)

    lst_vals  = clim_annual['mean_lst'].iloc[idx].values
    rain_vals = clim_annual['mean_rain'].iloc[idx].values

    jitter_results.append({
        'cluster_id' : int(row['cluster_id']),
        'true_lst'   : round(row['mean_lst'], 2),
        'lst_range'  : round(lst_vals.max()  - lst_vals.min(), 2),
        'rain_range' : round(rain_vals.max() - rain_vals.min(), 1),
    })

jitter_df = pd.DataFrame(jitter_results)
print("Jitter uncertainty — 100 simulated displacements per rural cluster (5 km radius):")
print(jitter_df.rename(columns={'lst_range': 'LST range (°C)', 'rain_range': 'Rain range (mm/mo)'}).to_string(index=False))
print()
print(f"Median LST  uncertainty (±):  {jitter_df['lst_range'].median() / 2:.2f} °C")
print(f"Median Rain uncertainty (±):  {jitter_df['rain_range'].median() / 2:.1f} mm/mo")
print()
print("Implication: even moderate positional jitter introduces measurable LST uncertainty,")
print("particularly near urban boundaries and water bodies where spatial gradients are steep.")

---
## Section 5 — Exercise 2 (3 pts): Join Births to Cluster-Level Annual Mean Climate

Merge `births` with `clim_annual` on `cluster_id` using a left join. Confirm there are no unmatched rows. Store the joined DataFrame as `births_env`.

Then compute: what fraction of births belong to clusters where the mean annual LST > 27 °C? Store as `frac_hot_clusters` (float, 3 decimal places).

In [ ]:
### BEGIN SOLUTION

births_env = births.merge(
    clim_annual[['cluster_id', 'mean_lst', 'mean_rain', 'mean_ndvi', 'mean_heat_index']],
    on='cluster_id',
    how='left'
)

unmatched = births_env['mean_lst'].isnull().sum()
print(f"Unmatched births (no climate record): {unmatched}")
assert unmatched == 0, "All births must match a cluster — check cluster_id values."

frac_hot_clusters = round(float((births_env['mean_lst'] > 27).mean()), 3)

print(f"Births joined successfully: {len(births_env):,} rows")
print(f"Fraction of births in clusters with mean annual LST > 27 °C: {frac_hot_clusters:.3f}")

### END SOLUTION

In [ ]:
### BEGIN HIDDEN TESTS
assert 'births_env' in dir(),                       "births_env DataFrame must be defined"
assert len(births_env) == len(births),               "births_env must have same number of rows as births"
assert 'mean_lst' in births_env.columns,             "births_env must contain mean_lst column"
assert births_env['mean_lst'].isnull().sum() == 0,   "No unmatched rows allowed after left join"
assert isinstance(frac_hot_clusters, float),         "frac_hot_clusters must be a float"
assert 0.0 < frac_hot_clusters < 1.0,               "frac_hot_clusters must be a proportion (0–1)"
### END HIDDEN TESTS

---
## Section 6 — Worked Example: Spatial Confounding — Facility Proximity and LBW

Clusters near health facilities may have higher care-seeking rates and more complete birth-weight recording. This creates spatial confounding that could distort any climate–LBW association. We stratify clusters by distance to the nearest facility as a diagnostic check.

In [ ]:
PROXIMITY_KM = 10.0

fac_tree = cKDTree(facilities[['latitude', 'longitude']].values)
_, fac_idx = fac_tree.query(cluster_coords, k=1)

dist_to_fac = haversine_approx_km(
    lat_arr,
    lon_arr,
    facilities['latitude'].values[fac_idx],
    facilities['longitude'].values[fac_idx]
)

clim_annual = clim_annual.copy()
clim_annual['dist_to_fac_km'] = dist_to_fac
clim_annual['near_facility']  = dist_to_fac <= PROXIMITY_KM

cluster_compare = (
    clusters[['cluster_id', 'lbw_rate']]
    .merge(clim_annual[['cluster_id', 'near_facility']], on='cluster_id')
)

near   = cluster_compare[cluster_compare['near_facility']]
remote = cluster_compare[~cluster_compare['near_facility']]

print(f"Clusters within {PROXIMITY_KM} km of a facility   (n={len(near):3d}):  "
      f"mean LBW rate = {near['lbw_rate'].mean():.3f}")
print(f"Clusters beyond {PROXIMITY_KM} km of a facility   (n={len(remote):3d}):  "
      f"mean LBW rate = {remote['lbw_rate'].mean():.3f}")
print()
print("Note: this stratification conflates access, care quality, and recording completeness.")
print("It is a spatial diagnostic, not a causal estimate.")

---
## Section 7 — Exercise 3 (2 pts): Cluster LBW Rate vs Mean Annual LST

Compute the Pearson correlation between cluster-level `lbw_rate` (from `zw_clusters.csv`) and mean annual `lst_day_c` (from the annual climate means). Store as `r_lbw_lst` (float, 3 decimal places).

Plot a scatter of the two variables with an OLS regression line.

In [ ]:
### BEGIN SOLUTION

cluster_joint = (
    clusters[['cluster_id', 'lbw_rate']]
    .merge(clim_annual[['cluster_id', 'mean_lst']], on='cluster_id')
)

r_lbw_lst = round(float(cluster_joint['lbw_rate'].corr(cluster_joint['mean_lst'])), 3)
print(f"Pearson r (cluster LBW rate vs mean annual LST): {r_lbw_lst:.3f}")

slope, intercept, *_ = stats.linregress(
    cluster_joint['mean_lst'], cluster_joint['lbw_rate']
)
x_line = np.linspace(cluster_joint['mean_lst'].min(), cluster_joint['mean_lst'].max(), 100)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    cluster_joint['mean_lst'], cluster_joint['lbw_rate'],
    s=18, alpha=0.5, color='steelblue', label='DHS cluster'
)
ax.plot(
    x_line, slope * x_line + intercept,
    color='firebrick', lw=2.0, label=f'OLS fit  (r = {r_lbw_lst})'
)
ax.set_xlabel('Mean annual LST (°C)', fontsize=11)
ax.set_ylabel('Cluster LBW rate', fontsize=11)
ax.set_title('Cluster-level LBW rate vs mean land surface temperature\nZimbabwe DHS 2015', fontsize=12)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.legend()
plt.tight_layout()
plt.show()

### END SOLUTION

In [ ]:
### BEGIN HIDDEN TESTS
assert isinstance(r_lbw_lst, float),     "r_lbw_lst must be a float"
assert -1.0 <= r_lbw_lst <= 1.0,         "correlation must lie in [-1, 1]"
assert abs(r_lbw_lst) < 0.85,            "implausibly strong correlation — check merge or variable names"
### END HIDDEN TESTS

---
## Recap

| Concept | Key point |
|---|---|
| `cKDTree` | O(log n) nearest-neighbour search; build once on grid coords, query many points |
| DHS jitter | 0–5 km rural, 0–2 km urban, ≤10 km for 1% of rural — coordinates are never exact |
| Jitter uncertainty | Monte Carlo: ±X °C LST and ±Y mm/mo rain uncertainty from positional error alone |
| Cluster-level join | Merge births to annual mean climate on `cluster_id`; individual-level spatial join is impossible |
| Spatial confounding | Facility proximity correlates with both LBW recording and local environment |

**Next:** Notebook 04 derives trimester-specific exposure windows from DHS birth dates, exploiting Zimbabwe's strong seasonal climate cycle to characterise *when* during pregnancy heat and drought occur.